# Objective Discovery with governed-rank

Not all policies are worth pursuing. Some objectives align with user preferences
— steering toward them helps everyone. Others fight user behavior — forcing them
destroys engagement.

This notebook shows how to use `govern()` as a **decision platform**: run candidate
policies through it to discover which objectives work BEFORE deploying anything.

**What you'll see:**
- A synthetic news catalog with engagement, popularity, and quality signals
- 7 candidate policies — from "promote trending" to "force diversity"
- Preference lift: do users actually want what each policy promotes?
- The gold mine: quality steering achieves engagement AND diversity simultaneously

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mosaic import govern

np.random.seed(42)
n = 500

# --- Categories with realistic distribution ---
cat_names = ['news', 'sports', 'business', 'culture', 'lifestyle', 'tech', 'opinion']
cat_probs = [0.30, 0.20, 0.10, 0.08, 0.12, 0.12, 0.08]
categories = np.random.choice(cat_names, size=n, p=cat_probs)

# --- Engagement (base model) ---
# Slight category bias: news/sports are more "engaging" on average.
# This means the base ranking over-represents popular categories.
cat_boost = {'news': 0.08, 'sports': 0.06, 'business': 0.02,
             'culture': -0.04, 'lifestyle': 0.0, 'tech': 0.03, 'opinion': -0.06}
engagement = np.random.beta(2, 5, n) + np.array([cat_boost[c] for c in categories])
engagement = np.clip(engagement, 0.01, 0.99)

# --- Views (popularity) --- news/sports get more eyeballs
cat_views = {'news': 1.5, 'sports': 1.3, 'business': 0.8,
             'culture': 0.5, 'lifestyle': 0.7, 'tech': 1.1, 'opinion': 0.4}
views = (100 * np.array([cat_views[c] for c in categories])
         * (1 + 3 * engagement) * np.random.lognormal(0, 0.5, n)).astype(int)
views = np.clip(views, 1, 10000)

# --- Reading time (quality/depth) --- NOT driven by category
reading_time = np.random.gamma(3, 2, n) + 2 * engagement
reading_time = np.clip(reading_time, 0.5, 30)

# --- Freshness ---
hours_ago = np.clip(np.random.exponential(24, n), 0.5, 72)

print(f'Catalog: {n} articles, {len(cat_names)} categories')
for c in cat_names:
    print(f'  {c:<12} {(categories==c).sum():>3} articles')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.Set2(np.linspace(0, 1, len(cat_names)))
cat_to_color = {c: colors[i] for i, c in enumerate(cat_names)}

counts = [(categories == c).sum() for c in cat_names]
axes[0].barh(cat_names[::-1], counts[::-1], color=colors[::-1])
axes[0].set_xlabel('Articles')
axes[0].set_title('Category Distribution')

for c in cat_names:
    m = categories == c
    axes[1].scatter(np.log1p(views[m]), reading_time[m],
                    c=[cat_to_color[c]], alpha=0.4, s=15, label=c)
axes[1].set_xlabel('log(views)')
axes[1].set_ylabel('Reading Time (min)')
axes[1].set_title('Quality vs Popularity')
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print('News/sports cluster at higher views (right). Quality spans all categories.')

## 1. User Behavior as Ground Truth

We simulate 10,000 reading events. Users prefer engaging, popular, and quality
content — but they don't explicitly seek diversity or niche categories.

**Preference Lift** measures whether users disproportionately read articles from
a policy set:

```
Lift = P(article in Policy | user reads it) / P(article in Policy | catalog)
```

- Lift > 1.0: users prefer these articles (aligned policy)
- Lift < 1.0: users avoid these articles (misaligned policy)

In [ ]:
# User utility: engagement + popularity + quality
# Power-law scaling (exponent=3) makes preferences sharply non-linear:
# users strongly prefer the best content, creating dramatic lift differences.
utility = (0.2 * engagement / engagement.max() +
           0.4 * views / views.max() +
           0.4 * reading_time / reading_time.max()) ** 3

np.random.seed(123)
n_events = 10000
read_probs = utility / utility.sum()
read_events = np.random.choice(n, size=n_events, p=read_probs, replace=True)
read_counts = np.bincount(read_events, minlength=n)

print(f'Simulated {n_events:,} reads across {n} articles')
print(f'\nReads by category:')
for c in cat_names:
    m = categories == c
    rf = read_counts[m].sum() / n_events
    cf = m.sum() / n
    print(f'  {c:<12} reads={100*rf:.1f}%  catalog={100*cf:.0f}%  lift={rf/cf:.2f}x')

## 2. Candidate Policies

We test 7 policies spanning popularity, quality, freshness, category, and diversity.
For each, we compute preference lift to see if users actually want what the policy promotes.

In [ ]:
def pref_lift(mask):
    return (read_counts[mask].sum() / n_events) / (mask.sum() / n)

policies = {
    'trending':              views >= np.percentile(views, 90),
    'quality_depth':         reading_time >= np.percentile(reading_time, 75),
    'freshness_recent':      hours_ago <= 12,
    'topic_sports':          categories == 'sports',
    'topic_business':        categories == 'business',
    'diversity_underexposed': np.isin(categories, ['culture', 'opinion']),
    'longtail':              views <= np.percentile(views, 30),
}

lifts = {name: pref_lift(mask) for name, mask in policies.items()}
sorted_p = sorted(lifts.items(), key=lambda x: x[1], reverse=True)

print(f'{"Policy":<25} {"Size":>5} {"Lift":>7} {"Aligned?":>9}')
print('-' * 50)
for name, lift in sorted_p:
    print(f'{name:<25} {policies[name].sum():>5} {lift:>6.2f}x {"YES" if lift > 1 else "no":>9}')

fig, ax = plt.subplots(figsize=(9, 5))
names = [p[0] for p in sorted_p]
vals = [p[1] for p in sorted_p]
bar_colors = ['#2ca02c' if v > 1 else '#d62728' for v in vals]
ax.barh(names[::-1], vals[::-1], color=bar_colors[::-1])
ax.axvline(x=1.0, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Preference Lift (>1 = aligned with users)')
ax.set_title('Which Policies Are Aligned with User Preferences?')
plt.tight_layout()
plt.show()

## 3. The Scorecard: govern() as Decision Platform

For each policy, we run `govern()` and measure two things:
- **Reads captured**: how many user reads are in the governed top-50 (engagement proxy)
- **Category entropy**: Shannon entropy of the category distribution in top-50 (diversity)

A good policy should improve reads captured (or hold steady) while maintaining
or increasing diversity.

In [ ]:
def cat_entropy(indices):
    cats = categories[list(indices)]
    _, counts = np.unique(cats, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + 1e-10))

base_scores = {i: float(engagement[i]) for i in range(n)}
base_order = sorted(range(n), key=lambda i: -engagement[i])
K = 50
base_entropy = cat_entropy(base_order[:K])
base_reads = sum(read_counts[i] for i in base_order[:K])

# Steer weight: moderate relative to engagement range (~0-0.75)
steer_weight = 0.15

results = []
for name, mask in policies.items():
    steering = {i: steer_weight * float(mask[i]) for i in range(n)}
    r = govern(base_scores, steering, budget=0.30)
    top_k = r.ranked_items[:K]
    ent = cat_entropy(top_k)
    reads = sum(read_counts[i] for i in top_k)
    results.append({
        'name': name, 'lift': lifts[name],
        'entropy': ent, 'd_ent': ent - base_entropy,
        'reads': reads, 'd_reads': reads - base_reads,
        'proj': r.projection_coeff
    })

results.sort(key=lambda x: x['lift'], reverse=True)

print(f'Base top-{K}: entropy={base_entropy:.3f}, reads captured={base_reads}')
print()
print(f'{"Policy":<25} {"Lift":>6} {"Entropy":>8} {"\u0394 Ent":>7} '
      f'{"Reads":>6} {"\u0394 Reads":>8} {"Proj":>7}')
print('-' * 72)
for r in results:
    print(f'{r["name"]:<25} {r["lift"]:>5.2f}x {r["entropy"]:>8.3f} '
          f'{r["d_ent"]:>+7.3f} {r["reads"]:>6} {r["d_reads"]:>+8} '
          f'{r["proj"]:>7.4f}')

## 4. The Gold Mine: Quality Beats Forced Diversity

The key finding: `quality_depth` (top 25% by reading time) is the **only** policy
that achieves both high preference lift AND positive diversity delta.

- **Trending**: users love it, but it concentrates the feed in news/sports
- **Diversity steering**: increases diversity, but users don't want the content
- **Quality**: users love it AND it naturally diversifies — because quality
  articles come from every category

You don't need to force diversity. Just surface quality content.

In [ ]:
key_policies = ['trending', 'quality_depth', 'diversity_underexposed']
key_results = {r['name']: r for r in results if r['name'] in key_policies}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: lift vs entropy delta (the sweet spot)
for r in results:
    color = '#2ca02c' if r['lift'] > 1 else '#d62728'
    size = 200 if r['name'] in key_policies else 60
    alpha = 1.0 if r['name'] in key_policies else 0.4
    axes[0].scatter(r['d_ent'], r['lift'], s=size, c=color, alpha=alpha, zorder=5)
    if r['name'] in key_policies:
        axes[0].annotate(r['name'], (r['d_ent'], r['lift']),
                         textcoords='offset points', xytext=(10, 5), fontsize=10,
                         fontweight='bold')
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.3)
axes[0].set_xlabel('Entropy \u0394 (positive = more diverse)')
axes[0].set_ylabel('Preference Lift (>1 = aligned)')
axes[0].set_title('Policy Discovery: The Sweet Spot')

# Right: category breakdown of top-50
bar_names = ['base'] + key_policies
width = 0.20
x = np.arange(len(cat_names))
for idx, name in enumerate(bar_names):
    if name == 'base':
        top_k = base_order[:K]
    else:
        steering = {i: steer_weight * float(policies[name][i]) for i in range(n)}
        r = govern(base_scores, steering, budget=0.30)
        top_k = r.ranked_items[:K]
    cc = [(categories[list(top_k)] == c).sum() for c in cat_names]
    axes[1].bar(x + idx * width, cc, width, label=name, alpha=0.8)

axes[1].set_xticks(x + 1.5 * width)
axes[1].set_xticklabels(cat_names, rotation=45, ha='right')
axes[1].set_ylabel(f'Articles in Top-{K}')
axes[1].set_title(f'Category Composition of Top-{K}')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

qd = key_results.get('quality_depth', {})
tr = key_results.get('trending', {})
du = key_results.get('diversity_underexposed', {})
print('KEY FINDING')
print('=' * 55)
print(f'quality_depth:          lift={qd["lift"]:.2f}x  \u0394ent={qd["d_ent"]:+.3f}  '
      f'\u0394reads={qd["d_reads"]:+d}')
print(f'trending:               lift={tr["lift"]:.2f}x  \u0394ent={tr["d_ent"]:+.3f}  '
      f'\u0394reads={tr["d_reads"]:+d}')
print(f'diversity_underexposed: lift={du["lift"]:.2f}x  \u0394ent={du["d_ent"]:+.3f}  '
      f'\u0394reads={du["d_reads"]:+d}')
print()
print('Quality is the ONLY policy with high lift AND positive entropy delta.')
print('It achieves diversity as a side effect of surfacing good content.')

## 5. Portfolio Optimization: Combining Policies

What if we mix policies? We can combine trending and quality steering signals
at different weights to explore the engagement-diversity frontier.

```python
steering = w_trending * trending_signal + w_quality * quality_signal
```

The question: is there a mix that gets BOTH high reads AND high diversity?

In [ ]:
trending_mask = policies['trending']
quality_mask = policies['quality_depth']

# Sweep: vary the trending/quality mix from 100% trending to 100% quality
weights = [(1.0, 0.0), (0.8, 0.2), (0.6, 0.4), (0.4, 0.6), (0.2, 0.8), (0.0, 1.0)]
mix_ent = []
mix_reads = []
mix_labels = []

for wt, wq in weights:
    steering = {i: steer_weight * (wt * float(trending_mask[i]) + wq * float(quality_mask[i]))
                for i in range(n)}
    r = govern(base_scores, steering, budget=0.30)
    top_k = r.ranked_items[:K]
    ent = cat_entropy(top_k)
    reads = sum(read_counts[i] for i in top_k)
    mix_ent.append(ent)
    mix_reads.append(reads)
    mix_labels.append(f'{int(100*wt)}/{int(100*wq)}')

fig, ax = plt.subplots(figsize=(8, 6))
for i, (ent, reads, label) in enumerate(zip(mix_ent, mix_reads, mix_labels)):
    color = plt.cm.RdYlGn(i / (len(weights) - 1))
    ax.scatter(ent, reads, s=150, c=[color], zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(label, (ent, reads), textcoords='offset points',
                xytext=(8, 5), fontsize=9)

ax.scatter(base_entropy, base_reads, s=100, c='gray', marker='x', zorder=5, linewidth=2)
ax.annotate('base', (base_entropy, base_reads), textcoords='offset points',
            xytext=(8, -10), fontsize=9, color='gray')

ax.axhline(y=base_reads, color='gray', linestyle=':', alpha=0.3)
ax.axvline(x=base_entropy, color='gray', linestyle=':', alpha=0.3)
ax.set_xlabel('Category Entropy (higher = more diverse)')
ax.set_ylabel('Reads Captured in Top-50 (higher = better engagement)')
ax.set_title('Portfolio Frontier: Trending/Quality Mix\n(label = % trending / % quality)')
plt.tight_layout()
plt.show()

print(f'{"Mix":>8} {"Entropy":>9} {"\u0394 Ent":>7} {"Reads":>7} {"\u0394 Reads":>8}')
print('-' * 44)
for i, (wt, wq) in enumerate(weights):
    print(f'{mix_labels[i]:>8} {mix_ent[i]:>9.3f} {mix_ent[i]-base_entropy:>+7.3f} '
          f'{mix_reads[i]:>7} {mix_reads[i]-base_reads:>+8}')
print(f'{"base":>8} {base_entropy:>9.3f} {0:>+7.3f} {base_reads:>7} {0:>+8}')
print()
print('60/40 trending-quality mix captures the most reads (+686),')
print('but diversity drops. Only pure quality (0/100) achieves BOTH')
print('more reads AND higher diversity than the base ranking.')

## Summary

`govern()` is not just a reranking tool — it's a **policy experimentation platform**.

| Finding | Implication |
|---------|-------------|
| `trending` has high lift but lowers diversity | Popular content clusters in a few categories |
| `quality_depth` has high lift AND raises diversity | Quality content spans all categories |
| `diversity_underexposed` has low lift | Forcing diversity fights user preference |
| `longtail` has low lift | Users don't want unpopular content |

**Key insight**: You can achieve BOTH engagement AND diversity by steering toward
quality content rather than forcing diversity directly. Quality is the proxy for
good diversity.

**Strategic use**: Run this scorecard on your candidate policies BEFORE deploying.
Discover which objectives are aligned with users, and avoid costly failed A/B tests.

```python
from mosaic import govern

for policy_name, steering_signal in candidate_policies.items():
    result = govern(base_scores, steering_signal, budget=0.30)
    # measure lift, diversity, quality retention
    # decide which policies to deploy
```